# Kobo/XLSForm Survey Analysis Notebook

## Abstract

This notebook presents a reproducible workflow developed for a specific Kobo-based quantitative survey and anonymized for wider use. It integrates XLSForm metadata with response data so that skip logic, data-quality checks, derived variables, sampling decisions, descriptive analysis, optional inference, and reporting remain aligned with the survey instrument. The notebook improves the efficiency and consistency of quantitative survey analysis while keeping methodological decisions visible for review.

The original analytical approach is retained and anonymized. Sections marked **Survey-specific** identify decisions that must be reassessed before the workflow is applied to another survey, including eligibility rules, mappings, derived variables, sampling weights, breakdowns, indicators, and inferential methods.

## Contents

1. Environment setup
2. Kobo/XLSForm metadata and response data
3. Skip-logic-aware data-quality checks
4. Survey-specific derived variables
5. Sampling weights
6. Analysis and reporting functions
7. Exploratory data analysis
8. Survey-specific indicators
9. Descriptive statistics
10. Optional categorical and continuous inference
11. Excel report export

## Design and Sampling Strategy

The source survey used disproportionate stratified sampling across four strata to support subgroup analysis. Because sampling fractions differed across strata, post-stratification weights were applied to align each stratum’s contribution to descriptive estimates with its share of the target population.

> **Survey-specific:** Review the sampling design, weighting approach, and minimum subgroup bases before applying the notebook to another survey. The presence of strata alone does not necessarily require weighting.

## Statistical Methodology

The current analysis uses sampling-weighted descriptive estimates and survey-design-adjusted categorical inference. Comparisons use the configured weights, Province × Population Group strata, and available stratum population totals. Results are reported as Rao–Scott chi-square tests using the design-adjusted F approximation.

Tests require at least 30 valid responses overall, 10 respondents in each comparison group, and five responses in each outcome category. Comparisons that do not meet these thresholds are suppressed and reported descriptively. Weighted Phi is used for 2 × 2 tables and weighted Cramér's V for larger tables.

Continuous variables are summarized descriptively in the current survey. The notebook also retains optional standard continuous tests for adaptation: Welch's t-test or Mann–Whitney U for two groups, and Welch/one-way ANOVA or Kruskal–Wallis for three or more groups. These functions are unweighted and are not a substitute for design-adjusted continuous inference when a complex survey design must be incorporated.

Multiple-response options are assessed separately, and their percentages may sum to more than 100%.

> **Survey-specific alternatives:** Rao–Scott is appropriate only when design-adjusted inference is required and valid design information is available. Set `INFERENCE_MODE = "standard"` for a justified unweighted Pearson/Fisher analysis, or `INFERENCE_MODE = "none"` for descriptive analysis. Weighting and inference are separate decisions.

### Software

Python 3, pandas, NumPy, SciPy, statsmodels, openpyxl, matplotlib, and optional `svy`/`polars` for survey-design inference.

### Outputs

The notebook produces EDA figures and summary tables, overall and cross-tabulated results for configured breakdowns, indicator tables, and optional inferential results. The final Excel workbook presents these results in stacked reporting tables, including configurable Sex, Age, and Disability Disaggregation (SADD) and other survey-specific breakdowns.

## 1. Setup Environment
This section configures the working environment, including library imports and display settings.

In [ ]:
# Install dependencies once before opening the notebook:
# python3 -m pip install -r requirements.txt

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Setup complete.")

## 2. Load Survey Metadata and Response Data

- The XLSForm `survey` and `choices` sheets provide question types, sections, choice labels, and skip logic.
- The response workbook contains the analytical respondent-level dataset.

> Update the paths, label language, interview year, and preferred identifier before running the notebook with another survey.

In [ ]:
SURVEY_PATH = Path("data/survey.xlsx")
DATA_PATH = Path("data/responses.xlsx")
LABEL_COL = "label::English (en)"
INTERVIEW_YEAR = 2026
PREFERRED_ID_COLUMN = "Participant_ID"

survey_workbook = pd.ExcelFile(SURVEY_PATH)
if "survey" not in survey_workbook.sheet_names:
    raise ValueError("The XLSForm workbook must contain a 'survey' sheet.")

survey_df = pd.read_excel(survey_workbook, sheet_name="survey")
choices_df = (
    pd.read_excel(survey_workbook, sheet_name="choices")
    if "choices" in survey_workbook.sheet_names
    else pd.DataFrame()
)

if LABEL_COL not in survey_df.columns:
    print(f"Warning: '{LABEL_COL}' was not found; question names will be used as labels.")
if choices_df.empty:
    print("No choices sheet found; choice-based processing will be skipped when not applicable.")

print(f"Survey sheet: {survey_df.shape[0]} rows, {survey_df.shape[1]} columns")
print(f"Choices sheet: {choices_df.shape[0]} rows")

In [ ]:
from survey_utils.kobo_metadata import build_choice_maps


choice_maps = build_choice_maps(choices_df, LABEL_COL)
print(f"{len(choice_maps)} choice list(s) found.")

In [ ]:
from survey_utils.kobo_metadata import classify_question, get_list_name, parse_survey_structure


qmeta = parse_survey_structure(survey_df, LABEL_COL)
qmeta["analysis_type"] = qmeta["type"].apply(classify_question)
qmeta["list_name"] = qmeta["type"].apply(get_list_name)

# Exclude administrative and identifier variables from per-question analysis.
# These variables are retained only for data cleaning and processing steps.
ADMIN_VAR_CANDIDATES = {"enumerator_id", PREFERRED_ID_COLUMN}
qmeta = qmeta.loc[
    ~qmeta["name"].astype(str).str.lower().isin(
        {name.lower() for name in ADMIN_VAR_CANDIDATES}
    )
].reset_index(drop=True)

print("Data type distribution:")
print(qmeta["analysis_type"].value_counts())
print("Total:", qmeta["analysis_type"].value_counts().sum())

qmeta.head(15)

### Skip-Logic Interpreter

The XLSForm `relevant` column defines when a question is displayed and applicable. The notebook evaluates each question's inherited relevance chain so that legitimate skips are not reported as missing-data errors.

The interpreter supports equality and inequality, numeric comparisons, `selected()`, parentheses, and `and`/`or`/`not`. Unsupported functions generate a warning and conservatively treat the question as applicable.

In [ ]:
from survey_utils.kobo_metadata import compute_relevant_mask


RELEVANT_CHAIN_BY_VAR = dict(zip(qmeta["name"], qmeta["relevant_chain"]))

# Display the first question that contains skip-logic as a sanity check.
example_var = next(
    (var for var, chain in RELEVANT_CHAIN_BY_VAR.items() if chain),
    None,
)

print("Skip-logic interpreter initialized.")

if example_var is not None:
    print(f"Example `relevant_chain` for '{example_var}':")
    print(RELEVANT_CHAIN_BY_VAR[example_var])
else:
    print("No skip-logic expressions were found in the survey.")

In [ ]:
from survey_utils.analysis_utils import resolve_identifier_column


data = pd.read_excel(DATA_PATH)
RESPONSE_ID_COLUMN = resolve_identifier_column(data, PREFERRED_ID_COLUMN)
DUPLICATE_ID_COLUMN = resolve_identifier_column(data, PREFERRED_ID_COLUMN, fallback_column=None)
RESPONSE_ID_LABEL = RESPONSE_ID_COLUMN or "dataframe index"

if RESPONSE_ID_COLUMN is None:
    print("No configured identifier or Kobo _index found; DQ follow-up will use dataframe index.")
elif RESPONSE_ID_COLUMN == "_index":
    print("Configured identifier not found; DQ follow-up will use Kobo _index response references.")
elif RESPONSE_ID_COLUMN != PREFERRED_ID_COLUMN:
    print(f"Configured identifier '{PREFERRED_ID_COLUMN}' matched exported column '{RESPONSE_ID_COLUMN}'.")

print("Dataset dimensions:", data.shape)
print("\nSurvey sections:")
for section in qmeta["section"].unique():
    print(" -", section)
print("\nVariable type summary:")
print(qmeta["analysis_type"].value_counts())

### Kobo Export Value Harmonization (Label → Code)

Kobo exports may contain choice labels while skip logic and analytical rules use choice codes. This section maps recognized labels back to their codes using the XLSForm `choices` sheet. Unmatched values are preserved and listed for review.

In [ ]:
from survey_utils.kobo_metadata import build_choice_lookup, normalize_choice_value


LIST_NAME_BY_VAR = dict(zip(qmeta["name"], qmeta["list_name"]))

recoded_columns, recode_issues = [], []
for _, qrow in qmeta.iterrows():
    var = qrow["name"]
    # Process only single-choice variables that exist in the dataset and have an associated choice list.
    if var not in data.columns or qrow["analysis_type"] != "categorical_single" or not qrow["list_name"]:
        continue
    lookup = build_choice_lookup(choice_maps, qrow["list_name"])
    if not lookup:
        continue
    original = data[var]
    recoded = original.apply(lambda v: lookup.get(normalize_choice_value(v), np.nan) if pd.notna(v) else v)
    unmapped_mask = original.notna() & recoded.isna()
    if unmapped_mask.any():
        recode_issues.append((var, sorted(original[unmapped_mask].dropna().unique().tolist())))
    # Preserve the original value when no valid mapping is found.
    data[var] = recoded.where(~unmapped_mask, original)
    recoded_columns.append(var)

print(f"{len(recoded_columns)} single-choice (select_one) variables were harmonized from labels to codes.")
if recode_issues:
    print("\nWARNING: Unmapped values were detected in one or more choice lists:")
    for var, vals in recode_issues:
        print(f"  {var}: {vals}")
    print("The original values have been preserved. These records should be reviewed as potential data quality issues.")
else:
    print("All values were successfully mapped. No unmapped values were detected.")

## 3. Data Cleaning
This section performs a series of data-quality checks on the exported survey data. For each check, the notebook reports the number of affected records together with their configured response-reference values, allowing individual cases to be identified and reviewed directly.

Missing-value and consistency checks are evaluated according to the survey's skip-logic (`relevant`) conditions. Questions that are legitimately skipped because their `relevant` conditions are not satisfied are not treated as data-quality issues. Only records where an applicable question lacks a response, or a non-applicable question contains one, are flagged for review.

### 3.1 Initial Dataset

In [ ]:
n_initial = len(data)
print(f"Initial dataset size: {n_initial} records")

### 3.2 Eligibility Filter

This analysis includes only participants who provided informed consent. Records with `Q1 (Consent) = No` are excluded before any data-quality checks or statistical analyses are performed.

For transparency, the notebook reports the number of excluded records together with their configured response-reference values, and generates a summary table documenting the resulting change in dataset size.

> **Note**: The eligibility filter is survey-specific and is therefore intentionally hard-coded to the informed-consent question (`Q1`). When adapting this notebook to another Kobo survey, update the eligibility variable and exclusion criterion accordingly.

In [ ]:
# Exclude records where informed consent was not provided (Q1 = No).
from survey_utils.analysis_utils import record_identifiers

n_before_elig = len(data)
mask_excluded = data["q1"].astype(str).str.lower() == "no"
n_excluded_q1 = int(mask_excluded.sum())
excluded_ids_q1 = record_identifiers(data, mask_excluded, RESPONSE_ID_COLUMN)

print(f"Excluded records (Q1 = No): {n_excluded_q1}")
if excluded_ids_q1:
    print(f"{RESPONSE_ID_LABEL}: {excluded_ids_q1}")

data_stage1 = data[~mask_excluded].copy()
n_after_elig = len(data_stage1)

cleaning_flow = pd.DataFrame({
    "Step": ["Completed interviews", "Excluded (Q1 = No)", "After eligibility filter"],
    "Count": [n_before_elig, n_excluded_q1, n_after_elig],
})
cleaning_flow

### 3.3 Duplicate Check

This step identifies duplicate records based on the configured preferred response identifier. When duplicate participant IDs are detected, the first occurrence is retained and all subsequent records are removed from the analysis dataset.

For transparency, the notebook reports the duplicated preferred identifier values together with the records removed during deduplication. The data-cleaning summary table is then updated to reflect the final number of records retained for analysis.

In [ ]:
# Identify duplicate preferred identifiers and retain the first occurrence.
if DUPLICATE_ID_COLUMN:
    dup_id_values = data_stage1.loc[
        data_stage1.duplicated(subset=[DUPLICATE_ID_COLUMN], keep=False),
        DUPLICATE_ID_COLUMN,
    ].unique().tolist()
    dup_mask = data_stage1.duplicated(subset=[DUPLICATE_ID_COLUMN], keep="first")
    n_duplicates = int(dup_mask.sum())
    removed_ids = record_identifiers(data_stage1, dup_mask, RESPONSE_ID_COLUMN)
    print(f"Duplicate {DUPLICATE_ID_COLUMN} values: {dup_id_values}")
    print(f"Duplicate records removed (second and subsequent occurrences): {n_duplicates}")
    if removed_ids:
        print(f"Removed {RESPONSE_ID_LABEL} values: {removed_ids}")
else:
    dup_id_values = []
    dup_mask = pd.Series(False, index=data_stage1.index)
    n_duplicates = 0
    removed_ids = []
    print("Duplicate participant check skipped: no configured preferred identifier column is available.")

data_stage2 = data_stage1[~dup_mask].copy()
n_final = len(data_stage2)

cleaning_flow = pd.concat([
    cleaning_flow,
    pd.DataFrame({"Step": ["Duplicate preferred identifier removed", "Final analysis dataset"],
                  "Count": [n_duplicates, n_final]}),
], ignore_index=True)

cleaning_flow

### 3.4 Missing Value Analysis

Raw missing values are reported for every variable. The primary data-quality indicator, however, is whether a response is missing when the question is both applicable (`relevant`) and required (`required`).

For example, if `q3` is displayed only when `q1 (Consent) = 'yes'` and is marked `required = True`, a missing response from an eligible participant represents a genuine data-quality issue. In contrast, if `q45` ("Additional comments") is always displayed but marked `required = False`, respondents may legitimately leave it unanswered, and such missing values are not treated as data-quality issues.

Accordingly, the summary table reports the `Required` status together with `N_Relevant`, `Missing_Among_Relevant`, and `Unexpected_Missing_%` for each variable. Only missing responses to questions that are both applicable and required are flagged as genuine data-quality issues, together with their configured response-reference values. Missing responses to optional questions are reported in the summary table but are not flagged.

> **Note**: The variable names used above are drawn from the current survey and included for illustration. When adapting this notebook to another Kobo survey, the same logic applies regardless of the specific question names.

In [ ]:
missing_rows = []
unexpected_missing_ids = {}

for _, qrow in qmeta.iterrows():
    var = qrow["name"]
    if var not in data_stage2.columns:
        continue
    relevant_mask = compute_relevant_mask(data_stage2, qrow["relevant_chain"])
    # Evaluate the question's complete skip-logic chain to identify the records for which the question is applicable.
    n_relevant = int(relevant_mask.sum())
    missing_mask = data_stage2[var].isna()
    # A missing value is considered unexpected only when the question is applicable to the respondent.
    unexpected_missing_mask = relevant_mask & missing_mask
    n_unexpected = int(unexpected_missing_mask.sum())
    is_required = bool(qrow["required"])

    missing_rows.append({
        "Variable": var,
        "Required": is_required,
        "Missing_Count_Raw": int(missing_mask.sum()),
        "Missing_%_Raw": round(missing_mask.mean() * 100, 2),
        "N_Relevant": n_relevant,
        "Missing_Among_Relevant": n_unexpected,
        "Unexpected_Missing_%": round(n_unexpected / n_relevant * 100, 2) if n_relevant else np.nan,
    })
    # Flag only missing responses from questions that are both applicable and required.
    if n_unexpected > 0 and is_required:
        ids = record_identifiers(data_stage2, unexpected_missing_mask, RESPONSE_ID_COLUMN)
        unexpected_missing_ids[var] = ids

missing_summary = pd.DataFrame(missing_rows).set_index("Variable").sort_values(
    "Unexpected_Missing_%", ascending=False)

print("Summary for all variables (top 15 ranked by Unexpected_Missing_%):")
display(missing_summary.head(15))

real_issues = missing_summary[(missing_summary["Missing_Among_Relevant"] > 0) & (missing_summary["Required"])]
print(f"\nRequired variables with genuine missing data issues: {len(real_issues)}")
display(real_issues)

print(f"\n{RESPONSE_ID_LABEL} values for the affected records:")
for var, ids in unexpected_missing_ids.items():
    print(f"  {var}: {ids}")

### 3.5 Consistency Checks

This section performs two types of consistency checks.

**3.5.1 Skip-logic violations** are evaluated automatically for every question based on the survey's `relevant` conditions.

- **Unexpected missing**: the question is applicable (`relevant = True`) but no response is provided (see Section 3.4, Missing Value Analysis).
- **Answered but should be skipped**: the question is not applicable (`relevant = False`) but contains a response.

**3.5.2 Custom logical consistency checks** evaluate relationships between variables that cannot be inferred automatically from the Kobo form. These checks are defined manually according to the survey design and the analytical requirements.

For both types of checks, the notebook reports the affected response-reference values to facilitate record-level review.

In [ ]:
# ---- Skip-logic violations: Responses provided when the question should have been skipped ----
skip_violation_rows = []
skip_violation_ids = {}

for _, qrow in qmeta.iterrows():
    var = qrow["name"]
    # Skip questions that are always applicable.
    if var not in data_stage2.columns or not qrow["relevant_chain"]:
        continue
    relevant_mask = compute_relevant_mask(data_stage2, qrow["relevant_chain"])
    answered_mask = data_stage2[var].notna()
    violation_mask = (~relevant_mask) & answered_mask
    n_violation = int(violation_mask.sum())
    if n_violation > 0:
        ids = record_identifiers(data_stage2, violation_mask, RESPONSE_ID_COLUMN)
        skip_violation_rows.append({"Variable": var, "N_Answered_But_Should_Be_Skipped": n_violation})
        skip_violation_ids[var] = ids

skip_violation_summary = pd.DataFrame(skip_violation_rows).sort_values(
    "N_Answered_But_Should_Be_Skipped", ascending=False) if skip_violation_rows else pd.DataFrame(
    columns=["Variable", "N_Answered_But_Should_Be_Skipped"])

print("Variables with skip-logic violations (answered although not applicable):")
display(skip_violation_summary)
for var, ids in skip_violation_ids.items():
    print(f"  {var}: {RESPONSE_ID_LABEL} -> {ids}")

In [ ]:
# ---- Custom logical consistency checks ---------------------------------------
consistency_checks = {}
# Additional survey-specific consistency checks can be added here.

# Optional rule deliberately disabled: a household may have non-employment
# income (for example, remittances, pensions, savings, or assistance).
# mask_income = ((data_stage2["q10"].astype(str).str.lower() == "no") &
#                (pd.to_numeric(data_stage2["q11"], errors="coerce") > 0))
# consistency_checks["no_income_earner_but_income_gt0 (Q10=No & Q11>0)"] = mask_income

# The total number of children must be smaller than household size because the
# respondent is always an adult.
mask_children_ge_household = (
    pd.to_numeric(data_stage2["q7"], errors="coerce").fillna(0)
    + pd.to_numeric(data_stage2["q8"], errors="coerce").fillna(0)
    >= pd.to_numeric(data_stage2["q6"], errors="coerce")
)
consistency_checks[
    "children_count_greater_than_or_equal_to_household_size (Q7 + Q8 >= Q6)"
] = mask_children_ge_household

consistency_summary_rows = []
consistency_ids = {}
for label, mask in consistency_checks.items():
    count = int(mask.sum())
    consistency_summary_rows.append({"Check": label, "Flagged_Count": count,
                                     "Flagged_%": round(count / len(data_stage2) * 100, 2)})
    if count > 0:
        consistency_ids[label] = record_identifiers(data_stage2, mask, RESPONSE_ID_COLUMN)

consistency_summary = pd.DataFrame(consistency_summary_rows)
display(consistency_summary)
for label, ids in consistency_ids.items():
    print(f"  [{label}] {RESPONSE_ID_LABEL} -> {ids}")

### 3.6 Outlier Detection (IQR Method)

Potential outliers are identified for all continuous variables using the interquartile range (IQR) method. Continuous variables are detected automatically from the Kobo XLSForm based on the `integer` and `decimal` question types. For each variable, the notebook reports the number of valid observations (`N`), the observed minimum and maximum values, the calculated IQR-based lower and upper bounds, and the number of observations falling outside these limits.

The IQR method serves as a screening tool rather than an automatic exclusion criterion. Observations identified as outliers should be reviewed in the survey context before determining whether they represent data-entry errors, implausible values, or legitimate extreme observations.

> **Note**: Although the IQR method is applied automatically to all continuous variables, not every continuous variable is equally suited to outlier detection. Review the results before treating flagged observations as data-quality issues, particularly when adapting this notebook to a different survey.

In [ ]:
# Automatically identify all continuous variables (integer and decimal) defined in the Kobo XLSForm.
CONTINUOUS_VARS_FOR_OUTLIER_SCAN = (
    qmeta.loc[
        qmeta["analysis_type"] == "continuous",
        "name",].loc[lambda s: s.isin(data_stage2.columns)].tolist())

print(
    f"IQR screening will be performed for "
    f"{len(CONTINUOUS_VARS_FOR_OUTLIER_SCAN)} continuous variables.")

from survey_utils.analysis_utils import iqr_outlier_summary

outlier_summary = pd.DataFrame({v: iqr_outlier_summary(data_stage2[v])
                                 for v in CONTINUOUS_VARS_FOR_OUTLIER_SCAN if v in data_stage2.columns}).T
outlier_summary

#### Review of Identified Outliers

The notebook reports the configured response-reference values associated with each detected outlier to support record-level review. No modifications are applied automatically.

Several commonly used outlier treatment strategies are provided below as commented examples, including replacing outliers with missing values, imputing them using the mean, median, and Winsorizing values to the IQR boundaries. The appropriate approach depends on the survey context, the variable under consideration, and the analytical objectives. Consequently, the decision to treat or retain outliers is left to the analyst.

> **Note:** Outliers do not necessarily indicate data quality issues. Depending on the survey population and the variable being measured, extreme values may represent legitimate observations rather than errors. For this reason, the notebook identifies potential outliers but does not modify the dataset automatically.

In [ ]:
outlier_record_ids = {}
for v in CONTINUOUS_VARS_FOR_OUTLIER_SCAN:
    if v not in data_stage2.columns or v not in outlier_summary.index:
        continue
    s = pd.to_numeric(data_stage2[v], errors="coerce")
    lower, upper = outlier_summary.loc[v, "Lower_Bound"], outlier_summary.loc[v, "Upper_Bound"]
    mask = (s < lower) | (s > upper)
    ids = record_identifiers(data_stage2, mask, RESPONSE_ID_COLUMN)
    if ids:
        outlier_record_ids[v] = {"mask": mask, "ids": ids}
        print(f"{v}: {len(ids)} outlier -> {RESPONSE_ID_LABEL}: {ids}")

# ---------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------
# for v, info in outlier_record_ids.items():
#     mask = info["mask"]
#     s_numeric = pd.to_numeric(data_stage2[v], errors="coerce")
#     # 1) Replace the flagged values with missing values:
#     # data_stage2.loc[mask, v] = np.nan
#     # 2) Replace the flagged values with the variable median:
#     # data_stage2.loc[mask, v] = s_numeric.median()
#     # 3) Replace the flagged values with the variable mean:
#     # data_stage2.loc[mask, v] = s_numeric.mean()
#     # 4) Winsorize the flagged values to the IQR boundaries:
#     # lower, upper = outlier_summary.loc[v, ["Lower_Bound", "Upper_Bound"]]
#     # data_stage2.loc[mask, v] = s_numeric.clip(lower, upper)

In [ ]:
# Compile a summary of the data cleaning process.

total_unexpected_missing = sum(
    len(ids) for ids in unexpected_missing_ids.values()
)

total_skip_logic_violations = sum(
    len(ids) for ids in skip_violation_ids.values()
)

total_consistency_flags = sum(
    len(ids) for ids in consistency_ids.values()
)

total_outliers = sum(
    len(info["ids"]) for info in outlier_record_ids.values()
)

data_cleaning_summary = pd.DataFrame({
    "Metric": [
        "Initial records",
        "Excluded (eligibility filter)",
        "Duplicate records removed",
        "Final analysis dataset",
        "Variables with genuine missing data issues",
        "Records with genuine missing data issues",
        "Variables with skip-logic violations",
        "Records with skip-logic violations",
        "Survey-specific consistency checks",
        "Records flagged by consistency checks",
        "Variables with potential outliers",
        "Potential outlier records",
    ],
    "Value": [
        n_initial,
        n_excluded_q1,
        n_duplicates,
        n_final,
        len(real_issues),
        total_unexpected_missing,
        len(skip_violation_ids),
        total_skip_logic_violations,
        len(consistency_summary),
        total_consistency_flags,
        len(outlier_record_ids),
        total_outliers,
    ],
})
print("=" * 45)
print("Data Cleaning Summary")
print("=" * 45)
display(data_cleaning_summary)

In [ ]:
df = data_stage2.copy()
print(f"Final analysis dataset: {len(df)} records")

## 4. Derived Variables
This section creates the analytical variables used throughout the notebook. Derived variables improve consistency across descriptive analyses, statistical testing, and automated reporting.

> Survey-specific analytical definitions, such as administrative mappings and category thresholds, are grouped separately to facilitate adaptation when applying the notebook to a different Kobo survey.

### 4.1 Survey-Specific Configuration

The mappings, category thresholds, reference periods, and household-specific reference values below belong to the original survey methodology. Review and replace them before applying the notebook to another survey.

In [ ]:
# Survey-specific administrative mappings
province_codes = sorted(df["q2"].dropna().astype(str).str.strip().unique())
if len(province_codes) != 2:
    raise ValueError(
        "This anonymized configuration expects two province codes. "
        "Replace PROVINCE_MAP with the mapping required by the survey."
    )
PROVINCE_MAP = dict(zip(province_codes, ["Province 1", "Province 2"]))

population_choice_codes = list(
    choice_maps.get(LIST_NAME_BY_VAR.get("q3"), {}).keys()
)
if not population_choice_codes:
    raise ValueError("No choice codes were found for q3.")
POPULATION_GROUP_1_CODES = {population_choice_codes[0]}

AGE_GROUPS = [
    (18, 29, "Aged 18 to 29"),
    (30, 44, "Aged 30 to 44"),
    (45, 59, "Aged 45 to 59"),
    (60, 100, "Aged 60 and above"),
]
HH_SIZE_GROUPS = [
    (1, 5, "1 to 5 members"),
    (6, np.inf, "6 or more members"),
]
TOTAL_CHILDREN_GROUPS = [
    (0, 0, "No children"),
    (1, 1, "1 child"),
    (2, 2, "2 children"),
    (3, np.inf, "3 or more children"),
]

In [ ]:
# Survey-specific reference periods and household thresholds
FOOD_REFERENCE_DAYS = 7
MONTHLY_REFERENCE_DAYS = 30

MEB_TABLE = {
    (1, 3): 23164,
    (4, 4): 30802,
    (5, 5): 34440,
    (6, 6): 38078,
    (7, np.inf): 47966,
}

# Replace these values and document the source, currency, reference date, and
# household-size rule used by the new survey.

### 4.2 Administrative Variables

In [ ]:
df["Province"] = df["q2"].astype(str).str.strip().map(PROVINCE_MAP).fillna("Other")


def population_group(value):
    if pd.isna(value):
        return np.nan
    return (
        "Population Group 1"
        if str(value).strip() in POPULATION_GROUP_1_CODES
        else "Population Group 2"
    )


df["Population_Group"] = df["q3"].apply(population_group)
df["Stratum"] = df["Province"] + "-" + df["Population_Group"]

print("Distribution of analytical strata")
display(df[["Province", "Population_Group", "Stratum"]].value_counts())

In [ ]:
# ---- Household-size-specific MEB amount (Table 3 guidance) --------------------------

def required_meb(hh_size):
    """Return the applicable Minimum Expenditure Basket amount."""

    if pd.isna(hh_size):
        return np.nan

    for (lower, upper), amount in MEB_TABLE.items():
        if lower <= hh_size <= upper:
            return amount

    return np.nan

required_meb_amount = df["q6"].apply(required_meb)
df["Required_MEB"] = required_meb_amount

### 4.3 Demographic Variables

This subsection derives demographic variables used throughout the notebook. The survey-specific reporting categories are:

- **Respondent Age Group:** 18–29, 30–44, 45–59, and 60 years and above.
- **Household Size Group:** 1–5 members and 6 or more members.
- **Total Children Group:** no children, 1 child, 2 children, and 3 or more children.

Some variables require analyst-defined thresholds or assumptions. These decisions should be reviewed before applying the notebook to a different survey.

> No automatic correction is applied to implausible values. Any treatment should be based on the analytical objectives.


In [ ]:
df["Age"] = INTERVIEW_YEAR - pd.to_numeric(df["q4"], errors="coerce")


implausible_age = (df["Age"] < 18) | (df["Age"] > 100)
flagged_ids = record_identifiers(df, implausible_age, RESPONSE_ID_COLUMN)
print(f"Implausible age records flagged: {len(flagged_ids)}")
if flagged_ids:
    print(f"{RESPONSE_ID_LABEL}:", flagged_ids)

# -----------------------------------------------------------------------------
# Possible treatment options
#
# 1. Replace implausible values with missing values.
# df.loc[implausible_age, "Age"] = np.nan
# 2. Impute using the mean.
# df.loc[implausible_age, "Age"] = df.loc[~implausible_age, "Age"].mean()
# 3. Impute using the median.
# df.loc[implausible_age, "Age"] = df.loc[~implausible_age, "Age"].median()
# 4. Retain the original values if supported by field verification.
# -----------------------------------------------------------------------------

print("Age distribution summary")
display(df["Age"].describe())

In [ ]:
def age_group(age):
    """Categorize respondent age into analytical age groups.
    Age group boundaries are defined in the survey configuration section and should be reviewed before applying the notebook to a different survey."""

    if pd.isna(age):
        return np.nan
    for lower, upper, label in AGE_GROUPS:
        if lower <= age <= upper:
            return label
    return np.nan

df["Age_Group"] = df["Age"].apply(age_group)
print("Age group distribution")
display(df["Age_Group"].value_counts(dropna=False))

In [ ]:
# ---- Household Size ----------------------------------------------------------

def hh_size_group(size):
    """Categorize household size using the survey-specific analytical thresholds.
    Household size categories are defined in the survey configuration section and should be reviewed before applying the notebook to a different survey."""

    if pd.isna(size):
        return np.nan
    for lower, upper, label in HH_SIZE_GROUPS:
        if lower <= size <= upper:
            return label
    return np.nan

df["HH_Size_Group"] = df["q6"].apply(hh_size_group)

print("Household size distribution")
display(df["HH_Size_Group"].value_counts(dropna=False))

In [ ]:
# ---- Total Number of Children (Q7+Q8)-------------------------------------------

df["Total_Children"] = (pd.to_numeric(df["q7"], errors="coerce").fillna(0) +
                         pd.to_numeric(df["q8"], errors="coerce").fillna(0))


def total_children_group(children):
    """Categorize the total number of children using the survey-specific analytical thresholds.
    Child categories are defined in the survey configuration section and should be reviewed before applying the notebook to a different survey."""

    if pd.isna(children):
        return np.nan
    for lower, upper, label in TOTAL_CHILDREN_GROUPS:
        if lower <= children <= upper:
            return label
    return np.nan

df["Total_Children_Group"] = df["Total_Children"].apply(total_children_group)

print("Total children distribution")
display(df["Total_Children_Group"].value_counts(dropna=False))

In [ ]:
# ---- Total Expenditure --------------------------------------------------------
df["Total_Expenditure"] = pd.to_numeric(df["q34"], errors="coerce") * (MONTHLY_REFERENCE_DAYS / FOOD_REFERENCE_DAYS) + pd.to_numeric(df['q35'], errors="coerce") + pd.to_numeric(df['q36'], errors="coerce")

## 5. Sampling Weights

The current survey used disproportionate stratified sampling. Post-stratification weights are calculated within Province × Population Group strata as:

> **Sampling Weight = Population Size / Analytical Sample Size**

This restores the target population distribution for descriptive estimates. Displayed sample sizes remain unweighted respondent counts.

> **Survey-specific:** Set `WEIGHTING_MODE` according to the approved methodology. Use `post_stratified` for the current design, `precomputed` for an existing approved weight, or `none` when weighting is not required. The presence of strata alone does not automatically require weights.

In [ ]:
WEIGHTING_MODE = "post_stratified"  # post_stratified | precomputed | none
PRECOMPUTED_WEIGHT_COLUMN = None

POPULATION_BY_STRATUM = {
    "Province 1-Population Group 1": 186,
    "Province 1-Population Group 2": 741,
    "Province 2-Population Group 1": 46,
    "Province 2-Population Group 2": 185,
}

if WEIGHTING_MODE not in {"post_stratified", "precomputed", "none"}:
    raise ValueError("Invalid WEIGHTING_MODE.")

sample_n_by_stratum = df["Stratum"].value_counts().to_dict()
ANALYSIS_WEIGHT_COLUMN = None

if WEIGHTING_MODE == "post_stratified":
    missing_strata = set(df["Stratum"].dropna()) - set(POPULATION_BY_STRATUM)
    if missing_strata:
        raise ValueError(f"Population totals are missing for: {sorted(missing_strata)}")
    df["Sampling_Weight"] = df["Stratum"].map(
        lambda stratum: POPULATION_BY_STRATUM[stratum] / sample_n_by_stratum[stratum]
    )
    ANALYSIS_WEIGHT_COLUMN = "Sampling_Weight"

elif WEIGHTING_MODE == "precomputed":
    if not PRECOMPUTED_WEIGHT_COLUMN or PRECOMPUTED_WEIGHT_COLUMN not in df.columns:
        raise ValueError("Configure an existing PRECOMPUTED_WEIGHT_COLUMN.")
    weights = pd.to_numeric(df[PRECOMPUTED_WEIGHT_COLUMN], errors="coerce")
    if weights.isna().any() or (~np.isfinite(weights)).any() or (weights <= 0).any():
        raise ValueError("Precomputed weights must be finite, nonmissing, and positive.")
    df["Sampling_Weight"] = weights
    ANALYSIS_WEIGHT_COLUMN = "Sampling_Weight"

df["Stratum_Population_N"] = df["Stratum"].map(POPULATION_BY_STRATUM)
weight_summary = pd.DataFrame({
    "Population_N": pd.Series(POPULATION_BY_STRATUM, dtype=float),
    "Sample_n": pd.Series(sample_n_by_stratum, dtype=float),
})
weight_summary["Sampling_Weight"] = (
    weight_summary["Population_N"] / weight_summary["Sample_n"]
    if WEIGHTING_MODE == "post_stratified"
    else np.nan
)

print(f"Weighting mode: {WEIGHTING_MODE}")
display(weight_summary)

## 6. Analysis and Reporting Functions

The workflow separates reusable operations from survey-specific analytical decisions. The `survey_utils` folder contains the functions used for XLSForm metadata processing, relevance evaluation, descriptive tables, EDA, statistical testing, and Excel formatting. Configuration and methodological choices remain in the notebook at the stage where they are applied, allowing analysts to trace how survey logic is translated into analytical outputs and to adapt those choices without rewriting the supporting functions.

In [ ]:
from survey_utils.analysis_utils import (
    categorical_descriptives,
    categorical_group_percentages,
    categorical_report_table,
    continuous_descriptives,
    continuous_report_table,
    eda_categorical_summary,
    eda_continuous_summary,
    eda_multiple_summary,
    eda_weighting_label,
    indicator_report_table,
    multi_response_descriptives,
    multi_response_report_table,
)
from survey_utils.eda_utils import plot_categorical, plot_continuous, plot_multiple_response
from survey_utils.kobo_metadata import get_multi_binary


print("Reusable utilities loaded successfully.")

In [ ]:
print("Reusable analytical functions loaded successfully.")

## 7. Exploratory Data Analysis (EDA)

This section provides an overview of the analytical dataset prior to descriptive and inferential analysis. Its objective is to characterize the dataset, identify unusual patterns, evaluate variable distributions, and assess the data's suitability for subsequent statistical analyses.

Whenever possible, summaries are generated automatically from the Kobo survey metadata, allowing the notebook to adapt to different survey instruments with minimal manual modification. Unlike the descriptive statistics section, which produces report-ready outputs, this section is intended to support analytical decision-making and quality assessment.

### 7.1 Dataset Overview

`EDA_WEIGHT_COL` controls categorical and multiple-response EDA percentages. Use the active analysis weight or set it to `None` to inspect the unweighted sample distribution. Continuous EDA remains unweighted.

In [ ]:
# -----------------------------------------------------------------------------
# Dataset overview
# -----------------------------------------------------------------------------

analysis_summary = (
    qmeta["analysis_type"]
    .value_counts(dropna=False)
    .rename_axis("Analysis_Type")
    .reset_index(name="Variables")
)

dataset_summary = pd.DataFrame({
    "Metric": [
        "Observations",
        "Variables in analytical dataset",
        "Metadata-defined variables",
        "Derived variables"
    ],
    "Value": [
        len(df),
        df.shape[1],
        len(qmeta),
        df.shape[1] - len(qmeta)
    ]
})

print("Dataset overview")
display(dataset_summary)

print("Variable types")
display(analysis_summary)

In [ ]:
# Set to "Sampling_Weight" for weighted EDA percentages; use None for unweighted EDA.
# This setting affects exploratory displays only, not the standardized reporting outputs.
EDA_WEIGHT_COL = ANALYSIS_WEIGHT_COLUMN

DERIVED_CATEGORICAL = ["Province", "Population_Group", "Stratum", "Age_Group", "HH_Size_Group", "Total_Children_Group"]
DERIVED_CONTINUOUS = ["Age", "Total_Children", "Total_Expenditure"]
print(f"EDA weighting: {eda_weighting_label(EDA_WEIGHT_COL)}")

In [ ]:
EDA_OUTPUT_DIR = Path("eda_output")

CAT_OUTPUT_DIR = EDA_OUTPUT_DIR / "categorical"
CONT_OUTPUT_DIR = EDA_OUTPUT_DIR / "continuous"

CAT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MULTI_OUTPUT_DIR = EDA_OUTPUT_DIR / "multiple_response"

MULTI_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

### 7.2 Categorical Variables

This subsection reviews all analytical categorical variables identified from the survey metadata together with the derived categorical variables created during the analytical workflow. Its objective is to evaluate data completeness, inspect response distributions, and identify sparse categories or unusual response patterns before proceeding to weighted descriptive statistics and inferential analyses.

All summaries are generated using the reusable analytical functions defined in Section 6, ensuring a consistent analytical workflow throughout the notebook. The frequency tables and charts produced here follow the weighting basis set by `EDA_WEIGHT_COL` in Section 7.1; the resulting outputs are labeled accordingly and are intended solely to support analyst review and quality assessment, not report-ready reporting.

In [ ]:
# -----------------------------------------------------------------------------
# Categorical variables
# -----------------------------------------------------------------------------
metadata_categorical = qmeta.loc[qmeta["analysis_type"] == "categorical_single", "name"].tolist()
categorical_variables = [var for var in metadata_categorical + DERIVED_CATEGORICAL if var in df.columns]
categorical_summary_tables, categorical_frequency_tables = [], []
print(f"Categorical variables identified: {len(categorical_variables)} ({eda_weighting_label(EDA_WEIGHT_COL)})")

for var in categorical_variables:
    meta = qmeta.loc[qmeta["name"] == var]
    label = meta.iloc[0]["label"] if not meta.empty else "Derived variable"
    section = meta.iloc[0]["section"] if not meta.empty else "Derived Variables"
    summary = eda_categorical_summary(df, var, label, section, EDA_WEIGHT_COL)
    frequency = categorical_descriptives(df, var, LIST_NAME_BY_VAR, choice_maps, EDA_WEIGHT_COL)
    display(summary); display(frequency)
    categorical_summary_tables.append(summary)
    exported = frequency.rename_axis("Category").reset_index()
    exported.insert(0, "Variable", var); exported.insert(1, "Label", label); exported.insert(2, "Section", section)
    exported.insert(3, "Weighting", eda_weighting_label(EDA_WEIGHT_COL))
    categorical_frequency_tables.append(exported)
    plot_categorical(df, var, LIST_NAME_BY_VAR, choice_maps, title=label, output_dir=CAT_OUTPUT_DIR, weight_col=EDA_WEIGHT_COL)

categorical_summary = pd.concat(categorical_summary_tables, ignore_index=True) if categorical_summary_tables else pd.DataFrame()
categorical_frequency = pd.concat(categorical_frequency_tables, ignore_index=True) if categorical_frequency_tables else pd.DataFrame()
if not categorical_variables:
    print("No categorical variables available; categorical EDA skipped.")


### 7.3 Continuous Variables

This subsection reviews all analytical continuous variables identified from the survey metadata together with the derived continuous variables created during the analytical workflow.

For each variable, the notebook summarizes completeness, descriptive statistics, and the empirical distribution to identify skewed variables, unusual observations, and potential analytical limitations before proceeding to weighted descriptive statistics and inferential analyses.

Continuous-variable summaries remain unweighted regardless of the `EDA_WEIGHT_COL` setting introduced in Section 7.1, consistent with the descriptive statistics convention adopted for continuous measures throughout the notebook. The reported statistics therefore describe the observed sample directly.

In [ ]:
# -----------------------------------------------------------------------------
# Continuous variables
# -----------------------------------------------------------------------------
metadata_continuous = qmeta.loc[qmeta["analysis_type"] == "continuous", "name"].tolist()
continuous_variables = [v for v in metadata_continuous + DERIVED_CONTINUOUS if v in df.columns]
continuous_summary_tables, continuous_descriptive_tables = [], []

for var in continuous_variables:
    meta = qmeta.loc[qmeta["name"] == var]
    label = meta.iloc[0]["label"] if not meta.empty else "Derived variable"
    section = meta.iloc[0]["section"] if not meta.empty else "Derived Variables"
    summary = eda_continuous_summary(df, var, EDA_WEIGHT_COL)
    summary.insert(0, "Variable", var); summary.insert(1, "Label", label); summary.insert(2, "Section", section)
    descriptives = continuous_descriptives(df, var, weight_col=None)
    descriptives.insert(0, "Variable", var); descriptives.insert(1, "Label", label); descriptives.insert(2, "Section", section)
    descriptives.insert(3, "Weighting", eda_weighting_label(EDA_WEIGHT_COL))
    display(summary); display(descriptives)
    continuous_summary_tables.append(summary); continuous_descriptive_tables.append(descriptives)
    plot_continuous(df, var, title=label, output_dir=CONT_OUTPUT_DIR, weight_col=None)

continuous_summary = pd.concat(continuous_summary_tables, ignore_index=True) if continuous_summary_tables else pd.DataFrame()
continuous_eda_summary = pd.concat(continuous_descriptive_tables, ignore_index=True) if continuous_descriptive_tables else pd.DataFrame()
if not continuous_variables:
    print("No continuous variables available; continuous EDA skipped.")


### 7.4 Multiple-response Variables

This subsection reviews all multiple-response questions identified from the survey metadata.

Each response option is treated as an individual binary variable to summarize how frequently each option was selected across respondents. The resulting frequency distributions help identify dominant response patterns, infrequently selected options, and variables that may require careful interpretation during subsequent descriptive and inferential analyses.

Percentages are calculated using the number of respondents as the denominator and therefore represent the proportion of respondents selecting each option. As with the categorical summaries in Section 7.2, these percentages follow the weighting basis set by `EDA_WEIGHT_COL` in Section 7.1, and the resulting outputs are labeled accordingly.

In [ ]:
# -----------------------------------------------------------------------------
# Multiple-response variables
# -----------------------------------------------------------------------------
metadata_multiple = qmeta.loc[qmeta["analysis_type"] == "multiple_response"].copy()
multiple_summary_tables, multiple_frequency_tables = [], []
compatible_multiple = metadata_multiple.loc[
    metadata_multiple["name"].isin(df.columns)
    & metadata_multiple["list_name"].isin(choice_maps)
]
print(f"Compatible multiple-response questions: {len(compatible_multiple)} ({eda_weighting_label(EDA_WEIGHT_COL)})")

for _, question in compatible_multiple.iterrows():
    var, label, section, list_name = question[["name", "label", "section", "list_name"]]
    summary = eda_multiple_summary(choice_maps, list_name, EDA_WEIGHT_COL)
    frequency = multi_response_descriptives(df, var, list_name, choice_maps, EDA_WEIGHT_COL)
    if summary.empty or frequency.empty:
        continue
    summary.insert(0, "Variable", var); summary.insert(1, "Label", label); summary.insert(2, "Section", section)
    display(summary); display(frequency)
    multiple_summary_tables.append(summary)
    exported = frequency.copy(); exported.insert(0, "Variable", var); exported.insert(1, "Label", label); exported.insert(2, "Section", section)
    exported.insert(3, "Weighting", eda_weighting_label(EDA_WEIGHT_COL))
    multiple_frequency_tables.append(exported)
    plot_multiple_response(frequency, var, title=label, output_dir=MULTI_OUTPUT_DIR, weight_col=EDA_WEIGHT_COL)

multiple_response_summary = pd.concat(multiple_summary_tables, ignore_index=True) if multiple_summary_tables else pd.DataFrame()
multiple_response_frequencies = pd.concat(multiple_frequency_tables, ignore_index=True) if multiple_frequency_tables else pd.DataFrame()
if compatible_multiple.empty:
    print("No compatible multiple-response questions or choice metadata; multiple-response EDA skipped.")

### 7.5 EDA Summary Export

The exploratory data analysis results are exported to a structured Excel workbook to facilitate quality assurance, analytical review, and documentation.

The workbook contains separate worksheets for categorical, continuous, and multiple-response variables, including both variable-level summaries and detailed frequency or descriptive statistics. These outputs are intended for analyst review and should not be interpreted as report-ready survey results.

The categorical and multiple-response worksheets reflect the weighting basis set by `EDA_WEIGHT_COL` in Section 7.1, while the continuous worksheet remains unweighted regardless of that setting, consistent with Section 7.3. Each worksheet states its weighting basis explicitly.

In [ ]:
# -----------------------------------------------------------------------------
# Export EDA summary
# -----------------------------------------------------------------------------
EDA_SUMMARY_FILE = EDA_OUTPUT_DIR / "eda_summary.xlsx"
eda_exports = {"Read Me": pd.DataFrame({"Item": ["EDA weighting"], "Value": [eda_weighting_label(EDA_WEIGHT_COL)]})}
if not categorical_summary.empty:
    eda_exports.update({"Categorical Summary": categorical_summary, "Categorical Frequency": categorical_frequency})
if not continuous_summary.empty:
    eda_exports.update({"Continuous Summary": continuous_summary, "Continuous Descriptives": continuous_eda_summary})
if not multiple_response_summary.empty:
    eda_exports.update({"Multiple Response Summary": multiple_response_summary,
                        "Multiple Response Frequencies": multiple_response_frequencies})
with pd.ExcelWriter(EDA_SUMMARY_FILE, engine="openpyxl") as writer:
    for sheet_name, table in eda_exports.items():
        table.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"EDA summary exported to: {EDA_SUMMARY_FILE} ({eda_weighting_label(EDA_WEIGHT_COL)})")

## 8. Indicator Calculation

The indicator definitions and targets in the following cells are survey-specific. Review the source questions, response codes, respondent-level calculation, denominator, target, and reporting note before adapting them to another survey.

In [ ]:
# ---- Respondent-level indicator scores --------------------------------------

indicator_1_q19 = df["q19"].eq("yes").where(df["q19"].notna()).astype("Float64")
indicator_1_q20 = (
    df["q20"].isin(["somewhat_satisfied", "very_satisfied"])
    .where(df["q20"].notna())
    .astype("Float64")
)

# The two component questions have the same valid-response base in this survey.
# Keeping the check visible prevents a later form revision from silently turning
# a respondent score into an unequal-base average.
if not indicator_1_q19.notna().equals(indicator_1_q20.notna()):
    raise ValueError(
        "Indicator 1 component bases differ. Calculate the two component rates "
        "separately before averaging them."
    )

df["Indicator_1"] = (indicator_1_q19 + indicator_1_q20) / 2
df["Indicator_2"] = (
    df["q37"].isin(["greatly_met", "somewhat_met"])
    .where(df["q37"].notna())
    .astype("Float64")
)
df["Indicator_3"] = (
    df["q40"].eq("yes_agree")
    .where(df["q40"].notna())
    .astype("Float64")
)
df["Indicator_4"] = (
    df["Total_Expenditure"].ge(df["Required_MEB"])
    .where(df[["Total_Expenditure", "Required_MEB"]].notna().all(axis=1))
    .astype("Float64")
)

INDICATOR_LABELS = {
    "Indicator_1": "Indicator 1",
    "Indicator_2": "Indicator 2",
    "Indicator_3": "Indicator 3",
    "Indicator_4": "Indicator 4",
}

In [ ]:
from survey_utils.analysis_utils import calculate_indicator

In [ ]:
# ---- Overall indicator scorecard and reconciliation -------------------------

indicator_results = [
    calculate_indicator(df, "Indicator_1", "Indicator 1", 80, weight_col=ANALYSIS_WEIGHT_COLUMN),
    calculate_indicator(df, "Indicator_2", "Indicator 2", 80, weight_col=ANALYSIS_WEIGHT_COLUMN),
    calculate_indicator(df, "Indicator_3", "Indicator 3", 70, weight_col=ANALYSIS_WEIGHT_COLUMN),
    calculate_indicator(df, "Indicator_4", "Indicator 4", 80, weight_col=ANALYSIS_WEIGHT_COLUMN),
]
indicator_scorecard = pd.DataFrame(
    [result for result in indicator_results if result is not None]
)

q39_valid = df["q39"].notna()
q39_aware = q39_valid & df["q39"].eq("yes")
q40_valid = df["q40"].notna()
if not q40_valid.equals(q39_aware):
    raise ValueError(
        "Q40 valid responses do not exactly match respondents aware of the "
        "mechanism in Q39. Review skip logic before exporting Indicator 3."
    )

q39_weighted_awareness = (
    df.loc[q39_aware, ANALYSIS_WEIGHT_COLUMN].sum()
    / df.loc[q39_valid, ANALYSIS_WEIGHT_COLUMN].sum()
    * 100
    if ANALYSIS_WEIGHT_COLUMN
    else q39_aware.sum() / q39_valid.sum() * 100
)
indicator_scorecard["Reporting note"] = ""
indicator_scorecard.loc[
    indicator_scorecard["Indicator"].eq("Indicator 3"), "Reporting note"
] = (
    f"Conditional on awareness: Q39 weighted awareness "
    f"{q39_weighted_awareness:.1f}%; Q40 unweighted base n={int(q40_valid.sum())}."
)

expected_indicator_1 = (
    indicator_1_q19.mean() + indicator_1_q20.mean()
) / 2 * 100
observed_indicator_1 = indicator_scorecard.loc[
    indicator_scorecard["Indicator"].eq("Indicator 1"), "Unweighted (%)"
].iat[0]
if not np.isclose(expected_indicator_1, observed_indicator_1, atol=0.05):
    raise AssertionError("Indicator 1 does not reconcile to its two component rates.")

print("Indicator summary")
display(indicator_scorecard)

In [ ]:
plot_percentage_column = "Weighted (%)" if ANALYSIS_WEIGHT_COLUMN else "Unweighted (%)"

fig, ax = plt.subplots(figsize=(10, 3))
colors = ["#2E8B57" if met == "Yes" else "#C0392B" for met in indicator_scorecard["Target Achieved"]]
bars = ax.barh(indicator_scorecard["Indicator"],indicator_scorecard[plot_percentage_column],color=colors,)

# Target markers
ax.scatter(
    indicator_scorecard["Target (%)"],
    indicator_scorecard["Indicator"],
    marker="|",
    s=600,
    color="black",
    label="Target",
    zorder=3,
)

# Percentage labels
for bar, value in zip(bars, indicator_scorecard[plot_percentage_column]):
    ax.text(value + 1,bar.get_y() + bar.get_height() / 2,f"{value:.1f}%",va="center",fontsize=10,)

ax.set_xlim(0, 105)
ax.set_xlabel("Percentage (%)")
ax.set_ylabel("")
ax.set_title("Indicator Performance")
ax.legend(frameon=True)

plt.tight_layout()
plt.show()

## 9. Descriptive Statistics

This section produces report-ready overall and breakdown tables. Percentages and continuous estimates use `ANALYSIS_WEIGHT_COLUMN` when weighting is active and remain unweighted otherwise.

> **Survey-specific:** Review all reporting breakdowns, section assignments, open-text handling, and minimum bases. Reporting breakdowns are not necessarily sampling strata.

In [ ]:
# ------------------------------------------------------------------
# ANALYSIS AND REPORTING SETTINGS
# ------------------------------------------------------------------
PRIORITY_BREAKDOWNS = [
    "Province", "Population_Group", "q5", "Age_Group", "q9", "HH_Size_Group"
]
ADDITIONAL_BREAKDOWNS = ["Total_Children_Group"]
AVAILABLE_BREAKDOWNS = [
    variable
    for variable in PRIORITY_BREAKDOWNS + ADDITIONAL_BREAKDOWNS
    if variable in df.columns
]
BREAKDOWNS = AVAILABLE_BREAKDOWNS  
SELF_MAP = {"q2": "Province", "q3": "Population_Group", "q4": "Age_Group"}

CORE_REPORT_BREAKDOWNS = [
    variable
    for variable in [
        "Province", "Population_Group", "q5", "Age_Group", "q9",
        "HH_Size_Group", "Total_Children_Group",
    ]
    if variable in AVAILABLE_BREAKDOWNS
]
CONTINUOUS_REPORT_BREAKDOWNS = CORE_REPORT_BREAKDOWNS.copy()
INDICATOR_REPORT_BREAKDOWNS = [
    variable
    for variable in [
        "Province", "Population_Group", "q5", "Age_Group", "q9",
        "HH_Size_Group", "Total_Children_Group",
    ]
    if variable in AVAILABLE_BREAKDOWNS
]
REPORT_BREAKDOWNS_BY_SECTION = {
    "Consent": [],
    "Household Profile Section": [],
    "Timeliness Of Assistance Section": CORE_REPORT_BREAKDOWNS,
    "Quality of Cash Distribution Process": CORE_REPORT_BREAKDOWNS,
    "Accessibility": CORE_REPORT_BREAKDOWNS,
    "Utilization of Cash Assistance": CORE_REPORT_BREAKDOWNS,
    "Impact of Cash Assistance Section": CORE_REPORT_BREAKDOWNS,
    "Sufficiency of Assistance Section": CORE_REPORT_BREAKDOWNS,
    "Complaint and Feedback Mechanism": CORE_REPORT_BREAKDOWNS,
    "Unsectioned": [],
}
SECTION_BY_VAR = dict(zip(qmeta["name"], qmeta["section"]))
ANALYSIS_TYPE_BY_VAR = dict(zip(qmeta["name"], qmeta["analysis_type"]))

TEXT_DETAIL_PARENT = {
    "q2_1": "q2", "q3_1": "q3", "q13_1": "q13", "q17_1": "q17",
    "q26_1": "q26", "q27_1": "q27", "q28_1": "q28", "q31_1": "q31",
    "q32_1": "q32", "q37_1": "q37", "q38_1": "q38", "q41_2": "q41_1",
    "q42_1": "q42", "q44_1": "q44",
}
CODED_TEXT_QUESTIONS = {"q28_1", "q31_1", "q37_1", "q38_1", "q41_2", "q45"}
COUNT_ONLY_BASE_THRESHOLD = 10


def report_breakdowns_for(variable):
    if variable in TEXT_DETAIL_PARENT or variable in CODED_TEXT_QUESTIONS:
        return []
    section = SECTION_BY_VAR.get(variable)
    if ANALYSIS_TYPE_BY_VAR.get(variable) == "continuous":
        selected = CONTINUOUS_REPORT_BREAKDOWNS
        if section == "Household Profile Section":
            selected = [item for item in selected if item != "Province"]
    else:
        selected = REPORT_BREAKDOWNS_BY_SECTION.get(section, [])
    return [
        breakdown
        for breakdown in selected
        if breakdown != variable
        and SELF_MAP.get(variable) != breakdown
        and SELF_MAP.get(breakdown) != variable
    ]


descriptive_outputs = {"overall": {}, "breakdowns": {}}
print("Available breakdowns:", AVAILABLE_BREAKDOWNS)
print("Core report breakdowns:", CORE_REPORT_BREAKDOWNS)
print("Continuous report breakdowns:", CONTINUOUS_REPORT_BREAKDOWNS)

In [ ]:
# ------------------------------------------------------------------
# OVERALL DESCRIPTIVE TABLES
# ------------------------------------------------------------------

for _, row in qmeta.iterrows():
    var = row["name"]
    analysis_type = row["analysis_type"]
    if var not in df.columns:
        continue
    applicable = compute_relevant_mask(df, row["relevant_chain"])

    if analysis_type == "categorical_single":
        descriptive_outputs["overall"][var] = categorical_descriptives(
            df, var, LIST_NAME_BY_VAR, choice_maps,
            weight_col=ANALYSIS_WEIGHT_COLUMN, applicable_mask=applicable,
        )
    elif analysis_type == "continuous":
        descriptive_outputs["overall"][var] = continuous_descriptives(
            df.loc[applicable], var, weight_col=ANALYSIS_WEIGHT_COLUMN
        )
    elif analysis_type == "multiple_response":
        descriptive_outputs["overall"][var] = multi_response_descriptives(
            df, var, row["list_name"], choice_maps,
            weight_col=ANALYSIS_WEIGHT_COLUMN, applicable_mask=applicable,
        )
    elif analysis_type == "open_text" and (
        var in TEXT_DETAIL_PARENT or var in CODED_TEXT_QUESTIONS
    ):
        table = categorical_descriptives(
            df, var, LIST_NAME_BY_VAR, choice_maps,
            weight_col=ANALYSIS_WEIGHT_COLUMN, applicable_mask=applicable,
        )
        valid_n = int((applicable & df[var].notna()).sum())
        if valid_n < COUNT_ONLY_BASE_THRESHOLD and not table.empty:
            table = table.drop(
                columns=["Percent", "Weighted_Percent"], errors="ignore"
            )
        descriptive_outputs["overall"][var] = table

In [ ]:
# ------------------------------------------------------------------
# INDICATOR OVERALL TABLES
# ------------------------------------------------------------------

indicator_vars = list(INDICATOR_LABELS)
indicator_overall_source = df.assign(__Indicator_Overall="Overall")
for var in indicator_vars:
    descriptive_outputs["overall"][var] = indicator_report_table(
        indicator_overall_source,
        var,
        "__Indicator_Overall",
        weight_col=ANALYSIS_WEIGHT_COLUMN,
    )

In [ ]:
# ------------------------------------------------------------------
# SELECTED REPORT BREAKDOWN TABLES
# ------------------------------------------------------------------

for _, row in qmeta.iterrows():
    var = row["name"]
    analysis_type = row["analysis_type"]
    if var not in df.columns:
        continue
    applicable = compute_relevant_mask(df, row["relevant_chain"])
    for breakdown in report_breakdowns_for(var):
        if analysis_type == "categorical_single":
            table = categorical_report_table(
                df, var, breakdown, LIST_NAME_BY_VAR, choice_maps,
                weight_col=ANALYSIS_WEIGHT_COLUMN, applicable_mask=applicable,
            )
        elif analysis_type == "continuous":
            table = continuous_report_table(
                df, var, breakdown,
                weight_col=ANALYSIS_WEIGHT_COLUMN, applicable_mask=applicable,
            )
        elif analysis_type == "multiple_response":
            table = multi_response_report_table(
                df, var, breakdown, row["list_name"], choice_maps,
                weight_col=ANALYSIS_WEIGHT_COLUMN, applicable_mask=applicable,
            )
        else:
            continue
        descriptive_outputs["breakdowns"][(var, breakdown)] = table

In [ ]:
# ------------------------------------------------------------------
# INDICATOR BREAKDOWN TABLES
# ------------------------------------------------------------------

for var in indicator_vars:
    for breakdown in INDICATOR_REPORT_BREAKDOWNS:
        descriptive_outputs["breakdowns"][(var, breakdown)] = (
            indicator_report_table(
                df, var, breakdown, weight_col=ANALYSIS_WEIGHT_COLUMN
            )
        )

In [ ]:
# ------------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------------

print("=" * 80)
print("Descriptive Statistics Summary")
print("=" * 80)

print(f"Overall descriptive tables   : {len(descriptive_outputs['overall'])}")
print(f"Breakdown tables             : {len(descriptive_outputs['breakdowns'])}")

## 10. Inferential Statistics

The current survey uses `survey_design` mode: Rao–Scott categorical comparisons with the active sampling weight, Province × Population Group strata, and stratum population totals.

Alternative modes are retained for adaptation:

- `standard`: unweighted Pearson's chi-square or Fisher's exact test, subject to the configured assumptions and base thresholds;
- `none`: no inferential testing.

Continuous inference is disabled in the current survey. For another survey, `CONTINUOUS_INFERENCE_MODE = "standard"` activates the reusable unweighted test engine:

- two groups: Welch's t-test when normality is supported, otherwise Mann–Whitney U;
- three or more groups: Welch/one-way ANOVA when normality is supported, otherwise Kruskal–Wallis;
- effect sizes: Cohen's d, rank-biserial correlation, eta-squared, or epsilon-squared as applicable.

> **Survey-specific:** Select categorical and continuous inference separately from the sampling design, estimand, research questions, and available design information. Weighted descriptive estimates do not automatically require inference. The included continuous tests are standard unweighted tests; a complex design may require a different design-adjusted method.

### 10.1 Configuration and Statistical Test Engine

In [ ]:
# Survey-specific inference configuration.
INFERENCE_MODE = "survey_design"  # survey_design | standard | none
CONTINUOUS_INFERENCE_MODE = "none"  # standard | none
P_ADJUST_METHOD = None  # Example alternative: "fdr_bh"

STATISTICAL_CONFIG = {
    "alpha": 0.05,
    "min_sample_size": 30,
    "min_group_size": 10,
    "min_category_size": 5,
    "chi_square_min_expected": 5,
    "chi_square_max_lt5_percent": 20,
    "chi_square_min_expected_cell": 1,
}

SURVEY_DESIGN_CONFIG = {
    "weight_col": ANALYSIS_WEIGHT_COLUMN,
    "strata_cols": ["Province", "Population_Group"],
    "fpc_col": "Stratum_Population_N",
    "psu_col": None,
}


if INFERENCE_MODE not in {"survey_design", "standard", "none"}:
    raise ValueError("Invalid INFERENCE_MODE.")
if CONTINUOUS_INFERENCE_MODE not in {"standard", "none"}:
    raise ValueError("Invalid CONTINUOUS_INFERENCE_MODE.")
if INFERENCE_MODE == "survey_design" and not ANALYSIS_WEIGHT_COLUMN:
    raise ValueError(
        "survey_design inference requires an active weight. "
        "Configure weighting or select standard/none."
    )

from survey_utils.statistical_utils import (
    categorical_group_test,
    continuous_group_test,
    survey_categorical_group_test,
)


def run_categorical_comparison(data, variable, breakdown, **kwargs):
    kwargs.pop("config", None)
    if INFERENCE_MODE == "survey_design":
        return survey_categorical_group_test(
            data,
            variable,
            breakdown,
            config=STATISTICAL_CONFIG,
            **SURVEY_DESIGN_CONFIG,
            **kwargs,
        )
    if INFERENCE_MODE == "standard":
        return categorical_group_test(
            data,
            variable,
            breakdown,
            config=STATISTICAL_CONFIG,
            **kwargs,
        )
    result = {
        "variable": variable,
        "breakdown": breakdown,
        "test": None,
        "statistic": np.nan,
        "p_value": np.nan,
        "effect_size": np.nan,
        "effect_name": np.nan,
        "n": 0,
        "interpretation": "Statistical testing disabled",
        "status": "suppressed",
        "suppression_reason": "INFERENCE_MODE is set to 'none'.",
    }
    return (result, None) if kwargs.get("return_details") else result


print(f"Categorical inference mode: {INFERENCE_MODE}")
print(f"Continuous inference mode: {CONTINUOUS_INFERENCE_MODE}")

### 10.2 Statistical Analysis

The following cells apply the configured categorical test engine and, when enabled, the standard continuous test engine to eligible variables and selected breakdowns. Results are collected in a common structure for the Excel report.

In [ ]:
# ------------------------------------------------------------------
# ANALYSIS CONTAINERS
# ------------------------------------------------------------------

inferential_outputs = {
    "results": [],
    "details": {}
}

print("Analysis containers initialized.")

In [ ]:
# ------------------------------------------------------------------
# PRE-SPECIFIED CATEGORICAL ANALYSIS LOOP
# ------------------------------------------------------------------
for _, row in qmeta.iterrows():
    var, analysis_type = row["name"], row["analysis_type"]
    if var not in df.columns:
        continue
    breakdowns = report_breakdowns_for(var)
    if not breakdowns:
        continue
    applicable = compute_relevant_mask(df, row["relevant_chain"])
    eligible_data = df.loc[applicable].copy()

    if analysis_type == "categorical_single":
        for breakdown in breakdowns:
            result, details = run_categorical_comparison(
                eligible_data, var, breakdown, config=STATISTICAL_CONFIG,
                return_details=True,
                skip_self_comparison=(
                    var == breakdown
                    or SELF_MAP.get(var) == breakdown
                    or SELF_MAP.get(breakdown) == var
                ),
            )
            inferential_outputs["results"].append(result)
            if details is not None:
                inferential_outputs["details"][(var, breakdown)] = details

    elif analysis_type == "multiple_response":
        valid_parent = eligible_data[var].notna()
        for code, label in choice_maps.get(row["list_name"], {}).items():
            temp = eligible_data.loc[valid_parent].copy()
            binary_name = f"{var}::{label}"
            temp[binary_name] = get_multi_binary(temp, var, code).astype(int)
            for breakdown in breakdowns:
                result, details = run_categorical_comparison(
                    temp, binary_name, breakdown,
                    config=STATISTICAL_CONFIG, return_details=True,
                )
                result["variable"] = binary_name
                inferential_outputs["results"].append(result)
                if details is not None:
                    inferential_outputs["details"][(binary_name, breakdown)] = details

# Indicator 1 is composite and remains descriptive; Indicators 2–4 are binary.
for var in ["Indicator_2", "Indicator_3", "Indicator_4"]:
    for breakdown in INDICATOR_REPORT_BREAKDOWNS:
        result, details = run_categorical_comparison(
            df, var, breakdown, config=STATISTICAL_CONFIG, return_details=True
        )
        inferential_outputs["results"].append(result)
        if details is not None:
            inferential_outputs["details"][(var, breakdown)] = details

if CONTINUOUS_INFERENCE_MODE == "standard":
    for _, row in qmeta.loc[qmeta["analysis_type"] == "continuous"].iterrows():
        variable = row["name"]
        if variable not in df.columns:
            continue
        applicable = compute_relevant_mask(df, row["relevant_chain"])
        eligible_data = df.loc[applicable]
        for breakdown in report_breakdowns_for(variable):
            result, details = continuous_group_test(
                eligible_data,
                variable,
                breakdown,
                config=STATISTICAL_CONFIG,
                return_details=True,
            )
            inferential_outputs["results"].append(result)
            if details is not None:
                inferential_outputs["details"][(variable, breakdown)] = details

In [ ]:
# ------------------------------------------------------------------
# SUMMARY AND OPTIONAL MULTIPLE-COMPARISON ADJUSTMENT
# ------------------------------------------------------------------

from statsmodels.stats.multitest import multipletests

inferential_outputs["results"] = pd.DataFrame(inferential_outputs["results"])
inferential_outputs["results"]["adjusted_p_value"] = np.nan

if (
    P_ADJUST_METHOD
    and not inferential_outputs["results"].empty
):
    completed = (
        inferential_outputs["results"]["status"].eq("completed")
        & inferential_outputs["results"]["p_value"].notna()
    )
    for _, indices in inferential_outputs["results"].loc[completed].groupby(
        "variable"
    ).groups.items():
        adjusted = multipletests(
            inferential_outputs["results"].loc[indices, "p_value"],
            method=P_ADJUST_METHOD,
        )[1]
        inferential_outputs["results"].loc[indices, "adjusted_p_value"] = adjusted

print("=" * 80)
print("Inferential Statistics Summary")
print("=" * 80)
print(f"Categorical mode          : {INFERENCE_MODE}")
print(f"Continuous mode           : {CONTINUOUS_INFERENCE_MODE}")
print(f"Comparison records        : {len(inferential_outputs['results'])}")
print(
    "Completed tests          :",
    int(inferential_outputs["results"]["status"].eq("completed").sum()),
)
print(
    "Suppressed comparisons   :",
    int(inferential_outputs["results"]["status"].eq("suppressed").sum()),
)
print(f"Diagnostic outputs        : {len(inferential_outputs['details'])}")

## 11. Excel Export

The Excel workbook uses stacked, Word-friendly question tables. Overall results can be followed by any configured reporting breakdown. This includes Sex, Age, and Disability Disaggregation (SADD), as well as geography, population group, household characteristics, or other survey-specific dimensions. Sample bases remain unweighted respondent counts.

> **Survey-specific:** Review report labels, breakdowns, open-text content, visual style, and output paths before distribution.

In [ ]:
# ------------------------------------------------------------------
# EXPORT CONFIGURATION
# Review visual identity, report wording, and visible breakdown selections.
# ------------------------------------------------------------------
OUTPUT_EXCEL_PATH = "Survey_Analysis_Report.xlsx"

REPORT_STYLE = {
    "primary_color": "3B3659",
    "secondary_color": "3B3659",
    "total_row_color": "F2F2F2",
    "text_on_primary": "FFFFFF",
    "border_color": "B7C9D6",
    "summary_fill_color": "FFFFFF",
}

BREAKDOWN_LABELS = {
    "Province": "Province",
    "Population_Group": "Population Group",
    "q5": "Gender",
    "Age_Group": "Age",
    "HH_Size_Group": "Household Size",
    "q9": "Disability",
    "Total_Children_Group": "Total Children Group",
}

REPORT_TEXT = {
    "section_suffix": "Survey Findings",
    "weighting_note": (
        f"Reported percentages and continuous estimates are {'weighted' if ANALYSIS_WEIGHT_COLUMN else 'unweighted'}. "
        "Sample sizes (n) show the actual number of respondents. Overall results "
        "are followed by configured breakdowns, including SADD where available. "
        "Statistical comparisons are shown only when enabled and eligible."
    ),
    "indicator_heading": "Indicator reporting tables",
}

from survey_utils.reporting_utils import (
    format_sheet,
    safe_sheet_name,
    write_stacked_question,
    write_section_heading,
)
from survey_utils.kobo_metadata import get_question_label
from openpyxl.styles import Alignment

print("Export configuration and report utilities loaded.")

In [ ]:
# Retain completed and explicitly suppressed pre-specified comparisons.
report_results = inferential_outputs["results"].copy()
report_results["_breakdown_order"] = report_results["breakdown"].map(
    {name: index for index, name in enumerate(AVAILABLE_BREAKDOWNS)}
)
report_results = (
    report_results
    .sort_values(["variable", "_breakdown_order"], kind="stable")
    .drop(columns="_breakdown_order")
    .reset_index(drop=True)
)

In [ ]:
# ------------------------------------------------------------------
# EXCEL EXPORT
# ------------------------------------------------------------------

used_sheet_names = {
    "Overview", "Sampling", "Cleaning", "Indicators",
    "Statistical Results", "Appendix",
}
population_total = weight_summary["Population_N"].sum()
sample_total = weight_summary["Sample_n"].sum()

statistical_export = report_results.copy()
if not statistical_export.empty:
    statistical_export.insert(
        1,
        "question_label",
        statistical_export["variable"].map(
            lambda variable: (
                INDICATOR_LABELS.get(variable)
                or get_question_label(qmeta, str(variable).split("::")[0])
            )
        ),
    )
    statistical_export["breakdown_label"] = statistical_export["breakdown"].map(
        lambda value: BREAKDOWN_LABELS.get(value, value)
    )
    statistical_columns = [
        "variable", "question_label", "breakdown_label", "n", "test",
        "statistic", "dof", "denominator_dof", "p_value",
        "effect_name", "effect_size", "effect_interpretation",
        "status", "interpretation", "suppression_reason",
    ]
    statistical_export = statistical_export.reindex(columns=statistical_columns)
    statistical_export.columns = [
        "Question Code", "Question", "Breakdown", "Valid n", "Test",
        "Test Statistic", "Numerator df", "Denominator df", "p-value",
        "Effect Size", "Effect Value", "Effect Interpretation",
        "Status", "Result", "Suppression Reason",
    ]

with pd.ExcelWriter(OUTPUT_EXCEL_PATH, engine="openpyxl") as writer:
    overview = pd.DataFrame({
        "Item": [
            "Project", "Analysis date", "Survey metadata source",
            "Response data source", "Population (N)", "Completed sample (n)",
            "Final analysis dataset (n)", "Survey questions", "Indicators",
            "Descriptive weighting", "Categorical inference",
            "Effect-size reporting", "Continuous inference",
            "Minimum test bases",
        ],
        "Value": [
            "Survey Analysis", pd.Timestamp.today().strftime("%Y-%m-%d"),
            str(SURVEY_PATH), str(DATA_PATH), population_total, sample_total,
            len(df), len(qmeta), len(indicator_vars),
            f"{'Sampling-weighted' if ANALYSIS_WEIGHT_COLUMN else 'Unweighted'} estimates; sample sizes (n) are unweighted",
            (
                "Rao–Scott chi-square (design-adjusted F) using Sampling_Weight, "
                "Province × Population Group strata, and stratum population totals"
                if INFERENCE_MODE == "survey_design"
                else f"{INFERENCE_MODE} mode; see notebook Section 10"
            ),
            (
                "Weighted Phi/Cramer's V is shown in Statistical Results for every "
                "eligible completed comparison, irrespective of significance."
            ),
            (
                (
                "Standard unweighted continuous tests enabled; see notebook Section 10"
                if CONTINUOUS_INFERENCE_MODE == "standard"
                else "Not included in this report; reusable standard tests remain available."
            )
            ),
            (
                f"Total n ≥ {STATISTICAL_CONFIG['min_sample_size']}; each group n ≥ "
                f"{STATISTICAL_CONFIG['min_group_size']}; each response category n ≥ "
                f"{STATISTICAL_CONFIG['min_category_size']}."
            ),
        ],
    })
    overview.to_excel(writer, sheet_name="Overview", index=False)
    weight_summary.reset_index(names="Stratum").to_excel(
        writer, sheet_name="Sampling", index=False
    )
    cleaning = pd.concat([
        cleaning_flow.assign(Output="Cleaning flow"),
        missing_summary.reset_index(names="Variable").assign(Output="Missing values"),
        outlier_summary.reset_index(names="Variable").assign(Output="Outlier screening"),
    ], ignore_index=True, sort=False)
    cleaning.to_excel(writer, sheet_name="Cleaning", index=False)
    indicator_scorecard.to_excel(writer, sheet_name="Indicators", index=False)
    statistical_export.to_excel(
        writer, sheet_name="Statistical Results", index=False
    )

    detail_children = set(TEXT_DETAIL_PARENT)
    details_by_parent = {}
    for detail, parent in TEXT_DETAIL_PARENT.items():
        details_by_parent.setdefault(parent, []).append(detail)

    for section, section_df in qmeta.groupby("section", sort=False):
        variables = [
            var for var in section_df["name"]
            if var in descriptive_outputs["overall"] and var not in detail_children
        ]
        if not variables:
            continue
        sheet_name = safe_sheet_name(section, used_sheet_names)
        ws = writer.book.create_sheet(sheet_name)
        writer.sheets[sheet_name] = ws
        write_section_heading(
            ws, f"{section} — {REPORT_TEXT['section_suffix']}",
            1, REPORT_STYLE,
        )
        ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=8)
        ws.cell(
            row=2, column=1, value=REPORT_TEXT["weighting_note"]
        ).alignment = Alignment(wrap_text=True)
        row_number = 4
        for variable in variables:
            row_number = write_stacked_question(
                ws,
                variable,
                row_number,
                analysis_type=ANALYSIS_TYPE_BY_VAR.get(variable, "categorical_single"),
                qmeta=qmeta,
                descriptive_outputs=descriptive_outputs,
                report_results=report_results,
                breakdowns=report_breakdowns_for(variable),
                breakdown_labels=BREAKDOWN_LABELS,
                alpha=STATISTICAL_CONFIG["alpha"],
                style=REPORT_STYLE,
                detail_variables=[
                    detail
                    for detail in details_by_parent.get(variable, [])
                    if detail in descriptive_outputs["overall"]
                ],
            )

    ws = writer.sheets["Indicators"]
    row_number = ws.max_row + 3
    write_section_heading(
        ws, REPORT_TEXT["indicator_heading"], row_number,
        REPORT_STYLE, size=12,
    )
    row_number += 1
    for variable in indicator_vars:
        row_number = write_stacked_question(
            ws,
            variable,
            row_number,
            analysis_type="indicator",
            qmeta=qmeta,
            descriptive_outputs=descriptive_outputs,
            report_results=report_results,
            breakdowns=INDICATOR_REPORT_BREAKDOWNS,
            breakdown_labels=BREAKDOWN_LABELS,
            alpha=STATISTICAL_CONFIG["alpha"],
            style=REPORT_STYLE,
            title_override=INDICATOR_LABELS[variable],
        )

    appendix = pd.DataFrame([
        {"Choice List": list_name, "Code": code, "Label": label}
        for list_name, mapping in choice_maps.items()
        for code, label in mapping.items()
    ])
    appendix.to_excel(writer, sheet_name="Appendix", index=False)
    for worksheet in writer.book.worksheets:
        format_sheet(worksheet)

print(f"Excel report completed: {OUTPUT_EXCEL_PATH}")